# 00 · 纯云端高速同步 MiniMax H3 满血模型 + 初始化独立项目目录到 5TB Google Drive

**极速零落盘架构（默认开启 Drive API Turbo）**：
- **绕过 DriveFS FUSE 瓶颈**：采用多线程 HTTP Range 滑动窗口预取 + **Google Drive v3 Resumable Upload 64MiB 内存流水线直传**，本地磁盘占用 **0 GB**（彻底免疫 107GB 免费 CPU 爆盘），支持多文件并发、断点续传与精确字节数 + LFS SHA256 校验。
- **全云端按项目隔离布局**：共享模型存 `/MyDrive/AI_Video/shared/models/`，各剧本项目独立存 `/MyDrive/AI_Video/projects/<PROJECT_ID>/`。
- **省算力提示**：本步请在右上角选 **免费 CPU 运行时（0 CU）** 执行，把全部 A100 80GB 计算单元留给 01/02 号 Notebook 生成视频。

In [ ]:
#@title 1. 挂载 5TB Google Drive、开启 Drive API Turbo 授权并拉取最新仓库代码
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}
USE_DRIVE_API_TURBO = True #@param {type:"boolean"}

from google.colab import drive
drive.mount('/content/drive')

if USE_DRIVE_API_TURBO:
    from google.colab import auth
    auth.authenticate_user()
    print('[Drive API Turbo] Google OAuth 授权完成，上传将自动绕过 DriveFS FUSE！')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q huggingface_hub hf_transfer hf_xet
!python -u {REPO_DIR}/scripts/sync_models_to_drive.py --manifest {REPO_DIR}/configs/models_manifest.json --packs dummy --list


In [ ]:
#@title 2. 高速同步共享模型包 + 将所有剧本/测试项目文件夹初始化到 5TB 云盘
PROJECT_ID = "01_rainy_signal" #@param {type:"string"}
MINIMAX_H3_FL2VA_BF16_FULL = True #@param {type:"boolean"}
MINIMAX_H3_REF2VA_BF16_FULL = False #@param {type:"boolean"}
MINIMAX_H3_TURBO_LORA = True #@param {type:"boolean"}
MINIMAX_H3_FL2VA_INT8_FULL = False #@param {type:"boolean"}
MINIMAX_H3_REF2VA_INT8_FULL = False #@param {type:"boolean"}
QWEN3VL_BF16_OPTIONAL = False #@param {type:"boolean"}
TRANSFER_MODE = "auto" #@param ["auto", "drive_api", "drivefs"]
FILE_WORKERS = 2 #@param {type:"integer"}
FETCH_WORKERS = 8 #@param {type:"integer"}
UPLOAD_CHUNK_MB = 64 #@param {type:"integer"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}
HF_TOKEN_INPUT = "" #@param {type:"string"}

import os, subprocess

hf_token = HF_TOKEN_INPUT.strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_token = ''

packs = []
if MINIMAX_H3_FL2VA_BF16_FULL: packs.append('minimax_h3_fl2va_bf16')
if MINIMAX_H3_REF2VA_BF16_FULL: packs.append('minimax_h3_ref2va_bf16')
if MINIMAX_H3_TURBO_LORA: packs.append('minimax_h3_turbo_lora')
if MINIMAX_H3_FL2VA_INT8_FULL: packs.append('minimax_h3_fl2va_int8_full')
if MINIMAX_H3_REF2VA_INT8_FULL: packs.append('minimax_h3_ref2va_int8_full')
if QWEN3VL_BF16_OPTIONAL: packs.append('qwen3vl_bf16_optional')

if not packs:
    raise ValueError('请至少勾选一个模型包！')

# 通过临时子进程环境变量兜底传递 OAuth Access Token（不落盘、不打印、不出现在命令行）
sub_env = os.environ.copy()
if TRANSFER_MODE != 'drivefs':
    try:
        import google.auth
        from google.auth.transport.requests import Request
        creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
        creds.refresh(Request())
        if getattr(creds, 'token', None):
            sub_env['GDRIVE_ACCESS_TOKEN'] = creds.token
    except Exception as exc:
        print(f'[提示] 未预刷新 Drive API Token ({exc})，脚本将自动尝试 ADC 或回退 DriveFS。')

cmd = [
    'python', '-u', f'{REPO_DIR}/scripts/sync_models_to_drive.py',
    '--repo-root', REPO_DIR,
    '--project', PROJECT_ID,
    '--manifest', f'{REPO_DIR}/configs/models_manifest.json',
    '--transfer-mode', TRANSFER_MODE,
    '--file-workers', str(FILE_WORKERS),
    '--fetch-workers', str(FETCH_WORKERS),
    '--upload-chunk-mb', str(UPLOAD_CHUNK_MB),
    '--packs', *packs,
]
if FORCE_REDOWNLOAD:
    cmd.append('--force')
if hf_token:
    cmd.extend(['--hf-token', hf_token])

print('执行命令:', ' '.join(cmd[:-2] + ['--hf-token', '***'] if hf_token else cmd))
subprocess.run(cmd, check=True, env=sub_env)


In [ ]:
#@title 3. 检查 5TB 云端共享模型库与独立项目目录结构
!echo '=== 共享模型库 (/MyDrive/AI_Video/shared/models) ==='
!du -sh /content/drive/MyDrive/AI_Video/shared/models/*
!find /content/drive/MyDrive/AI_Video/shared/models -type f -exec ls -lh {} +
!echo '\n=== 独立项目目录 (/MyDrive/AI_Video/projects) ==='
!find /content/drive/MyDrive/AI_Video/projects -maxdepth 3
